# Condition D literature preparation pilot
Select a T4 GPU runtime, then run all cells in order. No Hugging Face token is required for these public models. This runs three chunking batches, not diagnosis. The papers remain pending passage approval; no diagnostic index is built here.


In [1]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

project = Path('/content/drive/Othercomputers/My Laptop (1)/CP5403')
print("Drive connected:", Path('/content/drive/MyDrive').is_dir())
print("Project found:", (project / 'Files/phm_literature.py').is_file())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive connected: True
Project found: True


In [2]:
from pathlib import Path
import os, sys, subprocess, json, hashlib
from google.colab import drive
PROJECT_ROOT = Path('/content/drive/Othercomputers/My Laptop (1)/CP5403')
# Mount state is independent of the project path. Never remount merely
# because Othercomputers or a synced project is temporarily unavailable.
DRIVE_ROOT = Path('/content/drive')
if not (DRIVE_ROOT / 'MyDrive').is_dir():
    try:
        drive.mount(str(DRIVE_ROOT))
    except ValueError as exc:
        raise RuntimeError(
            'Google Drive mount failed before project setup. In Colab select '
            'Runtime > Disconnect and delete runtime, reconnect, then rerun '
            'this cell and complete Google Drive authorisation. '
            'Do not delete /content/drive or repeatedly force-remount it.'
        ) from exc
if not (PROJECT_ROOT / 'Files/phm_literature.py').is_file():
    raise FileNotFoundError(
        f'Drive is mounted, but the project is not available at {PROJECT_ROOT}. '
        'Use the Colab Files panel to locate CP5403 and update PROJECT_ROOT. '
        'Remounting is not a fix for a missing project path.'
    )
os.chdir(PROJECT_ROOT)
SOURCE = PROJECT_ROOT / 'Datasets/Literature-RAG'
OUTPUT = PROJECT_ROOT / 'Datasets/PHM15-literature-D-v1'
POLICY = PROJECT_ROOT / 'Files/literature_sources.json'
registry = json.loads(POLICY.read_text(encoding='utf-8'))['sources']
pdfs = sorted(SOURCE.glob('*.pdf'))
assert len(pdfs) == 8, f'Expected 8 PDFs; found {len(pdfs)}. Finish Drive sync first.'
for pdf in pdfs:
    digest = hashlib.sha256(pdf.read_bytes()).hexdigest()
    assert digest in registry, f'Unregistered or unsynced PDF: {pdf.name}'
    assert not registry[digest].get('excluded'), f'Excluded source: {pdf.name}'
assert 'searchable_scan' in (PROJECT_ROOT / 'Files/phm_literature.py').read_text(encoding='utf-8'), 'Updated extraction code has not synced.'
print('Verified all eight registered PDFs. Project:', PROJECT_ROOT)


Verified all eight registered PDFs. Project: /content/drive/Othercomputers/My Laptop (1)/CP5403


In [3]:
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU, then rerun.'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(PROJECT_ROOT / 'Files/requirements-literature-colab.txt')], check=True)
print('Dependencies installed. Continue to the next cell.')


GPU: Tesla T4
Dependencies installed. Continue to the next cell.


In [4]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'Files',
                '-p', 'test_phm_literature.py', '-q'], check=True)
PREPARE = [sys.executable, 'Files/phm_literature.py', '--sources', str(SOURCE),
           '--output', str(OUTPUT), '--policy', str(POLICY)]
subprocess.run(PREPARE, check=True)
contract = json.loads((OUTPUT / 'preparation_contract.json').read_text(encoding='utf-8'))
assert len(contract['sources']) == 8
for audit in contract['audits']:
    assert audit['request_count'] > 0, f"No extractable prose for {audit['document_id']}"
print('Prepared all eight sources. Total chunking batches:', sum(a['request_count'] for a in contract['audits']))


Prepared all eight sources. Total chunking batches: 199


In [5]:
# Three boundary-selection batches only. Successful responses are cached.
subprocess.run([sys.executable, 'Files/phm_literature_qwen.py', '--output', str(OUTPUT),
                '--max-requests', '3'], check=True)
# Revalidate Qwen boundaries against source text and save the latest report.
subprocess.run(PREPARE, check=True)
contract = json.loads((OUTPUT / 'preparation_contract.json').read_text(encoding='utf-8'))
print('Validated Qwen batches:', sum(a['qwen_validated_requests'] for a in contract['audits']))
print('Chunk preparation methods:', contract['candidate_methods'])
print('Outputs:', OUTPUT)
print('Latest report pointer:', PROJECT_ROOT / 'Outputs/LATEST_RUN.json')
print('STOP HERE. Let Drive sync, then ask for review. Passage approval and indexing remain pending.')


Validated Qwen batches: 1
Chunk preparation methods: {'paragraph_fallback': 1732, 'paragraph_fallback_context_join': 288, 'qwen_boundaries_context_join': 1, 'qwen_boundaries': 2}
Outputs: /content/drive/Othercomputers/My Laptop (1)/CP5403/Datasets/PHM15-literature-D-v1
Latest report pointer: /content/drive/Othercomputers/My Laptop (1)/CP5403/Outputs/LATEST_RUN.json
STOP HERE. Let Drive sync, then ask for review. Passage approval and indexing remain pending.
